# **Evaluasi Final di Data Test**

Notebook ini adalah **satu-satunya tempat data test dipakai**. Semua keputusan sudah dikunci di notebook sebelumnya, yaitu:
- preprocessing dan pembagian data (notebook 02)
- arsitektur dan hyperparameter (notebook 03, 04, dan 04b)
- loss dan threshold (notebook 05)
- metode penjelasan (notebook 06)

Di notebook ini tidak ada lagi pemilihan apa pun berdasarkan data test. Angka yang dihasilkan di sini adalah angka yang dilaporkan sebagai performa akhir model, baik di laporan Tugas 1 maupun UTS.

Isi notebook:
1. Memastikan model dan konfigurasi yang dievaluasi sama dengan yang dibekukan.
2. Metrik lengkap pada threshold final, beserta interval kepercayaan 95% (bootstrap).
3. Perbandingan dengan dan tanpa fitur unik di data test.
4. Pemeriksaan generalisasi (train, validation, test) dan kalibrasi probabilitas.
5. Analisis kesalahan prediksi: responden prediabetes, kelompok usia dan jenis kelamin, serta profil *false negative*.
6. Ringkasan penjelasan Integrated Gradients di data test.

**Penanggung jawab:** Felix Malvin (231111970)

## **1. Import Library dan Konfigurasi**

In [1]:
import sys
import json
import hashlib
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from scipy.stats import spearmanr
from sklearn.metrics import (classification_report, precision_recall_curve, roc_curve,
                             brier_score_loss, average_precision_score, roc_auc_score)
from sklearn.calibration import calibration_curve

from src import config
from src import preprocessing as prep
from src import evaluation as ev
from src import inference as inf
from src import visualization as viz

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 170)
viz.set_style()
WARNA = {"utama": "#2a78d6", "kedua": "#eb6834", "ketiga": "#1baf7a", "abu": "#9A9A9A"}
rng = np.random.default_rng(config.SEED)
print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


## **2. Memastikan Model yang Dievaluasi Adalah Model yang Dibekukan**

Sidik jari (SHA-256) file model dan konfigurasi dicatat sebagai bukti file mana yang dievaluasi. Prediksi model di data validation juga dicocokkan dengan probabilitas yang tersimpan di notebook 05. Kalau ada yang berbeda, berarti model berubah setelah threshold dipilih dan evaluasi ini tidak sah.

In [2]:
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

data = prep.prepare_datasets()
prep_cfg = prep.load_preprocessing_config()
assert data["test_fingerprint"] == prep_cfg["split"]["test_fingerprint"], "Data test berbeda dengan notebook 02"

model_cfg = json.loads((config.MODELS_DIR / "model_config.json").read_text(encoding="utf-8"))
MODEL_PATH = ROOT / model_cfg["model_path"]
THRESHOLD = model_cfg["threshold"]
model = keras.models.load_model(MODEL_PATH, compile=False)

prob_val = inf.predict_proba(model, data["X_val"])
prob_val_05 = np.load(config.RESULTS_DIR / "05_prob_validation_seed42.npz")["final"]
y_val = data["y_val"].astype(int)
y_test = data["y_test"].astype(int)

print("Fingerprint data test      :", data["test_fingerprint"])
print("Model                      :", model_cfg["model_path"])
print("SHA-256 model              :", sha256(MODEL_PATH)[:16])
print("SHA-256 model_config.json  :", sha256(config.MODELS_DIR / "model_config.json")[:16])
print("Konfigurasi                :", model_cfg["konfigurasi"])
print("Loss                       :", model_cfg["loss"], "| threshold:", THRESHOLD, f"({model_cfg['kriteria_threshold']})")
print("Prediksi validation sama dengan notebook 05:", np.allclose(prob_val, prob_val_05, atol=1e-6))
print(f"Data test: {len(y_test):,} responden, {y_test.sum():,} diabetes ({y_test.mean():.2%})")

Fingerprint data test      : 01b58633cb6f22b5
Model                      : models/mlp_final.keras
SHA-256 model              : dbb016120af20624
SHA-256 model_config.json  : c896a6ae59631952
Konfigurasi                : {'hidden_units': [128, 64, 32, 16], 'dropout': 0.0, 'l2': 1e-06, 'batch_norm': True, 'learning_rate': 0.0043925333029596605, 'batch_size': 256}
Loss                       : BCE | threshold: 0.13 (F2 maksimum di data validation)
Prediksi validation sama dengan notebook 05: True
Data test: 34,332 responden, 5,253 diabetes (15.30%)
